[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/master/examples/eval_ball_detector.ipynb)

# Ball detector eval — acceptance gate on bake-off clip (Phase 2)

Runs both the baseline roboflow ball detector and the fine-tuned model
against `data/bakeoff_clip.mp4` (via Drive mount) and reports the
sustained-detection rate for each.

**Acceptance gate:** the fine-tuned model must achieve ≥75% sustained ball
detection on the bake-off clip. If the gate fails, return to the labeling
runbook and label additional frames focused on the specific failure modes
(typically: ball at far touchline, fast motion, occlusion).

Place the fine-tuned weights at `MyDrive/soccer-vision/ball_yolov8_v1.pt`
before running.

In [ ]:
!pip install -q "ultralytics>=8.2" gdown

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
INPUT_CLIP = Path("/content/drive/MyDrive/soccer-vision/bakeoff_clip.mp4")
FINETUNED = Path("/content/drive/MyDrive/soccer-vision/ball_yolov8_v1.pt")
BASELINE = Path("/content/baseline_ball.pt")

assert INPUT_CLIP.exists(), f"Place bakeoff_clip.mp4 at {INPUT_CLIP}"
assert FINETUNED.exists(), f"Place fine-tuned weights at {FINETUNED}"

In [ ]:
import gdown
import cv2
from ultralytics import YOLO
from tqdm import tqdm

# Download roboflow's baseline ball model (one-time)
if not BASELINE.exists():
    gdown.download(
        "https://drive.google.com/uc?id=1isw4wx-MK9h9LMr36VvIWlJD6ppUvw7V",
        str(BASELINE),
        quiet=False,
    )

baseline_model = YOLO(str(BASELINE)).to("cuda")
finetuned_model = YOLO(str(FINETUNED)).to("cuda")


def detection_rate(model: YOLO, clip: Path) -> float:
    """Fraction of frames in `clip` where the model produces at least one ball detection."""
    cap = cv2.VideoCapture(str(clip))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    hits = 0
    for _ in tqdm(range(total), desc=str(model.ckpt_path)):
        ok, frame = cap.read()
        if not ok:
            break
        r = model(frame, imgsz=1280, verbose=False, conf=0.25)[0]
        if len(r.boxes) > 0:
            hits += 1
    cap.release()
    return hits / total


baseline_rate = detection_rate(baseline_model, INPUT_CLIP)
finetuned_rate = detection_rate(finetuned_model, INPUT_CLIP)

In [ ]:
print(f"Baseline ball detection rate:  {baseline_rate * 100:.1f}%")
print(f"Fine-tuned ball detection rate: {finetuned_rate * 100:.1f}%")
print(f"Improvement: +{(finetuned_rate - baseline_rate) * 100:.1f} pts")
print()
gate_passed = finetuned_rate >= 0.75
print(f"Acceptance gate (≥75%): {'PASS' if gate_passed else 'FAIL — label more frames'}")